# ChEMBL36 Pretraining Dataset Statistics

## Setup

In [ ]:
from setup import *

from matplotlib.ticker import MaxNLocator

## Compute Dataset Statistics

In [ ]:
non_property_keys = {"coord", "numbers", "charges", "id", "fingerprint", "ptr", "edge_index", "edge_attr",
                     "carbnyl"}  # carbnyl: stale duplicate of carbonyl from an older postprocess run (identical values)
paths = sorted(CHEMBL_DIR.glob("*.pt"))
properties, element_counts, n_conformers = {}, np.zeros(119, dtype=np.int64), set()
for path in tqdm(paths, desc="Loading dataset properties"):
    data = torch.load(path, map_location="cpu", weights_only=False, mmap=True)
    numbers = data["numbers"].numpy()
    n_conformers.add(data["coord"].shape[1])
    element_counts += np.bincount(numbers.ravel(), minlength=119)
    derived = {"num_atoms": np.full(len(numbers), numbers.shape[1]),
               "heavy_atoms": (numbers != 1).sum(1),
               "net_charge": data["charges"].numpy().astype(np.int64).sum(1)}
    for key, values in derived.items():
        properties.setdefault(key, []).append(values)
    for key, values in data.items():
        if key not in non_property_keys: properties.setdefault(key, []).append(values.numpy())
properties = {key: np.concatenate(values) for key, values in properties.items()}
num_molecules = len(properties["num_atoms"])
print(f"{len(paths)} files, {num_molecules:,} molecules, {n_conformers} conformers per molecule, {len(properties)} properties")

size_statistics = {
    key: {"median": np.median(properties[key]), "max": properties[key].max()}
    for key in ("num_atoms", "heavy_atoms")
}
property_statistics = {
    key: {"mean": values.mean(), "median": np.median(values),
          "min": values.min(), "max": values.max(),
          "fraction nonzero": np.mean(values != 0)}
    for key, values in properties.items() if key not in size_statistics
}

## Figure S1: ChEMBL36 Dataset Statistics

In [ ]:
statistics_dir = SRC_DIR / "statistics"
statistics_dir.mkdir(parents=True, exist_ok=True)
fig_size = (5.7 * cm, 5.7 * cm)
spectral = plt.get_cmap("Spectral")
category_colors = {"size": spectral(0.0), "rings": spectral(0.2), "functional group": spectral(0.9)}

def style_ax(ax, title):
    ax.set_title(title, fontsize=6, pad=2)
    ax.tick_params(labelsize=4, length=2, width=0.4, pad=1)
    ax.tick_params(which="minor", length=1, width=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for spine in ["left", "bottom"]:
        ax.spines[spine].set_linewidth(0.4)

def count_axis(ax, log_y):
    """y axis of molecule (or atom) counts: log when one bin dominates, else linear with a shared 10^k offset."""
    if log_y:
        ax.set_yscale("log")
        ax.set_ylim(bottom=0.5)  # bins holding a single molecule stay visible
    else:
        ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
        ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
        ax.yaxis.get_offset_text().set_fontsize(4)

def property_histogram(values, title, xlabel, color, path, log_y):
    """Histogram of one per-molecule property. Integer-valued properties get one bar per integer (with a gap between
    bars when there are few, and at least min_span integer slots so a 0/1 property does not fill the axis).
    log_y for counts dominated by zero (e.g. almost no molecule has an azide), so the tail stays visible."""
    min_span = 8
    discrete = np.all(values == np.round(values))
    if discrete:
        lo, hi = values.min(), values.max()
        bins = np.arange(lo - 0.5, hi + 1.5)
        xlim = (lo - 0.5, max(hi, lo + min_span - 1) + 0.5)
    else:
        bins = np.histogram_bin_edges(values, bins=100)
        xlim = (bins[0], bins[-1])
    counts, _ = np.histogram(values, bins)
    fig, ax = plt.subplots(figsize=fig_size)
    ax.hist(bins[:-1], bins=bins, weights=counts, color=color, edgecolor="none",  # counts precomputed: fast for 2.7M values
            rwidth=0.8 if discrete and len(counts) <= 40 else 1.0)
    count_axis(ax, log_y)
    if discrete: ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xlim(*xlim)
    style_ax(ax, title)
    ax.set_xlabel(xlabel, fontsize=6, labelpad=1)
    ax.set_ylabel("Number of Molecules", fontsize=6, labelpad=2)
    fig.subplots_adjust(left=0.13, right=0.97, top=0.9, bottom=0.17)
    fig.savefig(path, dpi=600)
    plt.show()
    plt.close(fig)

special_properties = {  # name: (title, x label, color category, log y)
    "weight":         ("Molecular Weight", "Molecular weight (g/mol)", "size", False),
    "net_charge":     ("Net Formal Charge", "Net formal charge", "size", True),  # 99% neutral
    "rings_total":    ("Rings", "Number of rings", "rings", True),
    "rings_aromatic": ("Aromatic Rings", "Number of aromatic rings", "rings", True),
}  # functional-group counts all use a log y axis: most are zero for most molecules, and one scale makes them comparable
display_names = {"amine_primary": "Primary Amine", "amine_secondary": "Secondary Amine", "amine_tertiary": "Tertiary Amine", "n_oxide": "N-Oxide"}

with plt.rc_context({"mathtext.fontset": "custom", "mathtext.rm": plt.rcParams["font.family"][0]}):
    size_properties = {"num_atoms": ("Molecule Size (incl. H)", "Number of atoms"),
                       "heavy_atoms": ("Molecule Size (Heavy Atoms)", "Number of heavy atoms")}
    for key, (title, xlabel) in size_properties.items():
        property_histogram(properties[key], title, xlabel, category_colors["size"], f"{statistics_dir}/{key}.png", log_y=False)

    periodic_table = Chem.GetPeriodicTable()
    elements = [z for z in np.nonzero(element_counts)[0] if z != 1]
    fig, ax = plt.subplots(figsize=fig_size)
    ax.bar([periodic_table.GetElementSymbol(int(z)) for z in elements], element_counts[elements], width=0.7, color=category_colors["size"])
    count_axis(ax, log_y=True)  # C ~6e7 atoms vs As ~3e2: only a log axis shows every element
    style_ax(ax, "Element Frequency (Heavy Atoms)")
    ax.set_xlabel("Element", fontsize=6, labelpad=1)
    ax.set_ylabel("Number of Atoms", fontsize=6, labelpad=2)
    fig.subplots_adjust(left=0.13, right=0.97, top=0.9, bottom=0.17)
    fig.savefig(f"{statistics_dir}/element_frequency.png", dpi=600)
    plt.show()
    plt.close(fig)

    for key, values in properties.items():
        if key in size_properties: continue  # plotted above
        if key in special_properties: title, xlabel, category, log_y = special_properties[key]
        else: title, xlabel, category, log_y = display_names.get(key, key.replace("_", " ").title()), "Matches per molecule", "functional group", True
        property_histogram(values, title, xlabel, category_colors[category], f"{statistics_dir}/{key}.png", log_y)

## Molecular Size Summary

In [ ]:
display(pd.DataFrame.from_dict(size_statistics, orient="index").rename(index={
    "num_atoms": "Atoms (including H)", "heavy_atoms": "Heavy atoms",
}, columns={"median": "Median", "max": "Maximum"}))

## Element Counts

In [ ]:
display(pd.DataFrame({
    "Element": [Chem.GetPeriodicTable().GetElementSymbol(int(z)) for z in np.flatnonzero(element_counts)],
    "Atoms": element_counts[np.flatnonzero(element_counts)],
}))

## Molecular Property Summary

In [ ]:
with pd.option_context("display.max_rows", None):
    display(pd.DataFrame.from_dict(property_statistics, orient="index").rename_axis("property").round(4))